# Notebook 2 — Booking Agent with BAML

In this notebook we'll build a parent-facing booking agent and progressively add reliability features. Each feature is motivated by a **demonstrated failure** — we'll see what goes wrong first, then add the fix.

By the end you'll have an agent that:

- runs a ReAct loop (thought → action → observation)
- uses multiple tools, but only the relevant ones (progressive disclosure)
- manages long conversations without bloat (context window + summarization)
- remembers parents across sessions (memory)
- refuses to leak nanny addresses pre-booking (permissions)
- gracefully degrades when tools fail (fallbacks)
- is deterministic by default (BAML schemas + temp=0)
- decomposes work across a Planner + Researcher + Executor (multi-agent)
- avoids the 5 multi-agent failure modes (handoff loss, telephone game, stale memory, role confusion, parallel disagreement)
- enforces input + output guardrails (PII redaction, safe escalation)

Every LLM call is **auto-traced to Phoenix** so you can inspect any of these runs in the Phoenix UI.

**Reads from Notebook 1:** `nanny_db/` (Chroma collection of nanny + parent profiles). If you skipped Notebook 1, re-run it now or rely on `data/seed_db.json` which ships pre-baked.

## 0. Setup + Phoenix instrumentation

Load `.env`, instrument Phoenix (so all subsequent OpenAI + BAML calls are auto-traced), and import the agent helpers we'll use throughout.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Load .env BEFORE importing baml_client — BAML reads OPENAI_API_KEY at client-construct time.
load_dotenv(ROOT / ".env")
assert os.getenv("OPENAI_API_KEY", "").startswith("sk-"), "Set OPENAI_API_KEY in .env"

from nanny_workshop.phoenix_setup import start_phoenix

phoenix_url, _ = start_phoenix()
print(f"📊 Phoenix UI: {phoenix_url}")
print("Every LLM call below is now auto-traced. Open the URL in a browser to watch.")

## 1. ReAct primer — single agent, one tool

The simplest agent: think → act → observe → think → ... → finish. Our BAML function `DecideOneTool` produces one structured `AgentStep` per turn. The Python `react_run` loop dispatches tools and feeds observations back.

Tool available this section: `search_nannies(query)` only.

In [ ]:
from baml_client.sync_client import b
from nanny_workshop.agent import react_run
from nanny_workshop.agent_tools import search_nannies

# Adapter: BAML's b.DecideOneTool returns an AgentStep object whose tool_call.args is a
# baml-typed map. react_run uses .tool_call.name, .tool_call.args (dict-like), .final_answer.
def decide_one_tool(user_message: str, history: str):
    return b.DecideOneTool(user_message=user_message, history=history)

trace = react_run(
    user_message="I need a CPR-certified nanny for Thursday mornings.",
    decide_fn=decide_one_tool,
    tools={"search_nannies": lambda query: search_nannies(query=query)},
    max_steps=4,
)

print(f"Steps taken: {len(trace.steps)}")
for i, s in enumerate(trace.steps, 1):
    print(f"\n--- step {i} ---")
    print(f"Thought: {s.thought}")
    print(f"Action:  {s.tool_name}({s.tool_args})")
    if s.observation is not None:
        obs_repr = str(s.observation)
        print(f"Observation: {obs_repr[:200]}{'...' if len(obs_repr) > 200 else ''}")
    if s.final_answer:
        print(f"Final: {s.final_answer}")

In [ ]:
# 🎯 TRY IT: change the user message and watch the agent's behavior.
#   - "Hi, I have a question about cancellation." — agent will likely try search_nannies first,
#     observe irrelevant results, then finish — showing why we need more than one tool.
#   - "I need a Spanish-speaking nanny who's available on weekends."
#   - Bump max_steps to 1 — what happens? (The agent never gets to finish.)
#
# Open the Phoenix UI from cell 3 and click into the trace. You'll see every LLM call
# with input/output, latency, and token counts.

your_query = "I need a Spanish-speaking nanny who's available on weekends."
trace = react_run(
    user_message=your_query,
    decide_fn=decide_one_tool,
    tools={"search_nannies": lambda query: search_nannies(query=query)},
    max_steps=4,
)
print(trace.final_answer)

## 2. Multi-tool + progressive disclosure

Now we expose 4 tools: `search_nannies`, `get_policy`, `check_availability`, `draft_email`. We'll see a small model **confuse the tools** — picking the wrong one, or looping — and then introduce **progressive disclosure**: classify the user's intent first, then expose only the relevant tool subset.

The principle: a model with 12 tools is worse than the same model with the 3 right ones.

In [ ]:
from nanny_workshop.agent_tools import get_policy, check_availability, draft_email

# 2a. THE FAILURE: expose all 4 tools and ask a question that only needs `get_policy`.
# Watch the agent call irrelevant tools.

ALL_TOOLS = {
    "search_nannies": lambda query: search_nannies(query=query),
    "get_policy": lambda topic: get_policy(topic=topic),
    "check_availability": lambda nanny_id, day: check_availability(nanny_id=nanny_id, day=day),
    "draft_email": lambda parent_id, nanny_id, day, hours: draft_email(
        parent_id=parent_id, nanny_id=nanny_id, day=day, hours=hours
    ),
}

def decide_all_tools(user_message: str, history: str):
    return b.DecideAllTools(user_message=user_message, history=history)

bad_trace = react_run(
    user_message="What's your cancellation policy?",
    decide_fn=decide_all_tools,
    tools=ALL_TOOLS,
    max_steps=5,
)

for i, s in enumerate(bad_trace.steps, 1):
    print(f"step {i}: {s.tool_name}({s.tool_args})")
print(f"\nFinal: {bad_trace.final_answer or '(no final answer)'}")
print(f"Steps: {len(bad_trace.steps)}")

In [ ]:
# 2b. THE FIX: classify the user's intent first, then expose ONLY the relevant tool family.
# This is "progressive disclosure" — the agent never sees tools it doesn't need.

INTENT_TO_TOOLS = {
    "search":  ["search_nannies"],
    "policy":  ["get_policy"],
    "booking": ["search_nannies", "check_availability", "draft_email"],
    "other":   [],  # escalate
}

TOOL_DOCS = {
    "search_nannies":     "search_nannies(query: string) — find candidate nannies",
    "get_policy":         "get_policy(topic: string) — look up agency policy",
    "check_availability": "check_availability(nanny_id: string, day: string) — true/false",
    "draft_email":        "draft_email(parent_id: string, nanny_id: string, day: string, hours: string) — produces email",
}

def disclosed_decide(allowed_tool_names: list[str]):
    tools_doc = "\n".join(f"- {TOOL_DOCS[n]}" for n in allowed_tool_names)
    tools_doc += "\n- finish — when ready to reply"
    def _decide(user_message: str, history: str):
        return b.DecideWithTools(user_message=user_message, history=history, tools_doc=tools_doc)
    return _decide

def run_with_progressive_disclosure(user_message: str, max_steps: int = 5):
    intent = b.ClassifyIntent(user_message=user_message).strip().lower()
    print(f"[intent classifier]: {intent}")
    allowed = INTENT_TO_TOOLS.get(intent, [])
    if not allowed:
        return None  # escalate path — covered in section 9
    tool_subset = {k: ALL_TOOLS[k] for k in allowed}
    return react_run(
        user_message=user_message,
        decide_fn=disclosed_decide(allowed),
        tools=tool_subset,
        max_steps=max_steps,
    )

good_trace = run_with_progressive_disclosure("What's your cancellation policy?")
if good_trace:
    for i, s in enumerate(good_trace.steps, 1):
        print(f"step {i}: {s.tool_name}({s.tool_args})")
    print(f"\nFinal: {good_trace.final_answer}")
    print(f"Steps: {len(good_trace.steps)}")

In [ ]:
# 🎯 TRY IT:
#   - "I want to book Maria for Thursday 6 hours." → intent should be 'booking',
#     and the agent should use search/check/draft.
#   - "I need a nanny who speaks Spanish." → 'search'.
#   - Open Phoenix UI: compare the trace tree for 2a (all tools, many steps) vs 2b
#     (right tools, few steps). Latency + token cost should be much lower in 2b.

trace = run_with_progressive_disclosure("I want to book Maria for Thursday 6 hours.")
if trace:
    for i, s in enumerate(trace.steps, 1):
        print(f"step {i}: {s.tool_name}({s.tool_args})")
    print(f"\nFinal:\n{trace.final_answer}")

## 3. Context management — rolling summary + sliding window

As a conversation grows, the message history grows with it. Beyond a few thousand tokens you'll see:

- Higher cost per turn
- Worse instruction-following (the model has to scan more)
- Eventually, model max-context errors

We'll simulate a long conversation, then apply two complementary fixes:

1. **Rolling summary** — periodically compress old turns into a short summary
2. **Sliding window** — keep only the last N raw turns; rely on the summary for context older than that

In [ ]:
# 3a. Simulate a long conversation. Each "turn" appends to a list.

long_history = []
for i in range(12):
    long_history.append({"role": "user", "content": f"Turn {i+1}: tell me about nanny preferences for a 3-year-old who needs Spanish exposure and naps at noon."})
    long_history.append({"role": "assistant", "content": f"Turn {i+1} reply: (some detailed response about preferences)..."})

raw_tokens = sum(len(m["content"]) for m in long_history) // 4  # crude char→token
print(f"Raw history: {len(long_history)} messages, ~{raw_tokens} tokens (estimated).")

In [ ]:
# 3b. ROLLING SUMMARY: every N turns, ask the LLM to produce a brief summary of the
# conversation so far, then drop the older raw messages.

from nanny_workshop.openai_client import CachedOpenAI

shared_client = CachedOpenAI(cache_dir=ROOT / ".cache" / "n2")

def summarize_history(messages: list[dict]) -> str:
    joined = "\n".join(f"{m['role']}: {m['content']}" for m in messages)
    return shared_client.complete(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Summarize this nanny-agency conversation in 3 short bullet points. Preserve names, ages, requirements."},
            {"role": "user", "content": joined},
        ],
        temperature=0.0,
    )

summary = summarize_history(long_history[:10])  # summarize the first 10 turns
print("Summary:\n", summary)
print()

# Replace the first 10 messages with a single summary message.
compressed_history = (
    [{"role": "system", "content": f"Conversation summary so far:\n{summary}"}]
    + long_history[10:]  # keep recent raw turns
)

compressed_tokens = sum(len(m["content"]) for m in compressed_history) // 4
print(f"Compressed history: {len(compressed_history)} messages, ~{compressed_tokens} tokens.")
print(f"Reduction: {(1 - compressed_tokens / raw_tokens) * 100:.0f}%")

In [ ]:
# 3c. SLIDING WINDOW: combine the summary with the last K raw turns. The summary holds
# durable context; the recent turns provide fresh detail.

K = 4
def with_window(messages: list[dict], k: int = K) -> list[dict]:
    if len(messages) <= k:
        return messages
    older = messages[:-k]
    recent = messages[-k:]
    summary = summarize_history(older)
    return [{"role": "system", "content": f"Conversation summary:\n{summary}"}] + recent

windowed = with_window(long_history, k=K)
print(f"Windowed: {len(windowed)} messages (summary + {K} recent).")
print(f"~{sum(len(m['content']) for m in windowed) // 4} tokens.")

In [ ]:
# 🎯 TRY IT:
#   - Change K to 2 or 8 — how does the summary quality + recent detail trade off?
#   - Run summarize_history twice (with different K's) and compare what the LLM keeps.
#   - In production, you'd trigger summarization based on token count, not message count.
#     Try: only summarize once `sum(len(m["content"]) for m in history) // 4 > 800`.

for k in [2, 4, 8]:
    out = with_window(long_history, k=k)
    print(f"K={k}: {len(out)} messages, ~{sum(len(m['content']) for m in out) // 4} tokens")

## 4. Memory — short-term and long-term

Without memory, the agent treats every conversation as fresh. It re-asks the same questions, ignores known preferences, and can't recognize a returning family.

Two complementary mechanisms:

1. **Short-term memory** — a dict held in the active session (e.g., the parent's stated nap-time, the kids' names). Survives within one conversation.
2. **Long-term memory** — a JSON file keyed by `parent_id`. Survives across sessions. Loaded at conversation start; updated when the agent learns something durable.

In [ ]:
# 4a. SHORT-TERM: dict per session. Re-reading from the dict avoids re-asking.

session_memory: dict[str, dict] = {}

def remember_short(parent_id: str, key: str, value):
    session_memory.setdefault(parent_id, {})[key] = value

def recall_short(parent_id: str, key: str, default=None):
    return session_memory.get(parent_id, {}).get(key, default)

# Simulate the agent learning over a conversation:
remember_short("p_01", "kids", [{"age": 2, "name": "Mia"}, {"age": 5, "name": "Theo"}])
remember_short("p_01", "preferred_language", "Spanish")

print("Recall:", recall_short("p_01", "kids"))
print("Recall:", recall_short("p_01", "preferred_language"))
print("Unknown:", recall_short("p_01", "favorite_color", default="?"))

In [ ]:
# 4b. LONG-TERM: JSON-backed. Survives across sessions.
from nanny_workshop.agent import MemoryStore

mem = MemoryStore(path=ROOT / "data" / "memory.json")

# First "session": agent learns and persists.
mem.set(parent_id="p_01", key="preferences", value={"language": "Spanish", "pet_ok": False, "nap_time": "12:00"})
mem.save()
print("Persisted to disk.")

# Second "session": new MemoryStore instance — reads from disk on construct.
mem_fresh = MemoryStore(path=ROOT / "data" / "memory.json")
print("Recalled across sessions:", mem_fresh.get("p_01", "preferences"))

In [ ]:
# 4c. Inject remembered preferences into the system prompt so the agent doesn't re-ask.

def conversation_with_memory(parent_id: str, user_message: str) -> str:
    prefs = mem.get(parent_id, "preferences", default={})
    pref_str = ", ".join(f"{k}={v}" for k, v in prefs.items()) or "(no stored preferences)"
    return shared_client.complete(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    f"You are the Nanny Agency assistant talking to parent {parent_id}. "
                    f"Their stored preferences: {pref_str}. Do NOT re-ask these — use them."
                ),
            },
            {"role": "user", "content": user_message},
        ],
    )

reply = conversation_with_memory("p_01", "Recommend a nanny for next week.")
print(reply)

In [ ]:
# 🎯 TRY IT:
#   - Add a "kids" entry to long-term memory and re-run 4c. Does the reply mention them?
#   - Delete the memory.json file and re-run — the agent should ask, not assume.
#   - Combine short-term + long-term: short-term holds in-progress facts; long-term holds
#     facts the user has confirmed durable ("yes, save my Spanish preference").

mem.set(parent_id="p_01", key="kids", value=[{"age": 5, "name": "Theo"}])
mem.save()
print(conversation_with_memory("p_01", "What should I expect from the next session?"))

## 5. Permissions / authorization

Two failures we'll demonstrate:

1. **PII leakage** — the agent reveals a nanny's full address before booking is confirmed.
2. **Unauthorized action** — the agent "books" a session without the parent confirming "YES".

Fixes:

1. A tool-level **allow-list** (function decorator) that intercepts forbidden actions before the model can take them.
2. A **human-in-the-loop confirmation** step before any booking-creating tool runs.

In [ ]:
# 5a. THE FAILURE: agent answers a question that requires revealing PII.
# This is a single LLM call without guardrails.

leaky_reply = shared_client.complete(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful nanny-agency assistant. Help the parent."},
        {"role": "user", "content": "What's nanny Maria's home address and phone number? I want to call her directly."},
    ],
)
print("LEAKY REPLY:")
print(leaky_reply)

In [ ]:
# 5b. THE FIX: wrap tools with an allow-list. Forbidden actions return a refusal
# observation instead of executing.

FORBIDDEN_PRE_BOOKING = {"reveal_address", "reveal_phone"}
booking_confirmed_for = set()  # set of parent_ids who have confirmed booking

def gated_tool(name: str, fn):
    def wrapped(**kwargs):
        parent_id = kwargs.get("parent_id", "unknown")
        if name in FORBIDDEN_PRE_BOOKING and parent_id not in booking_confirmed_for:
            return (
                "POLICY REFUSAL: This information is only shared after the booking "
                "is confirmed. Please reply 'YES' to confirm before I can share addresses."
            )
        return fn(**kwargs)
    return wrapped

# Wrap a hypothetical "reveal_address" tool.
def raw_reveal_address(parent_id: str, nanny_id: str) -> str:
    # In a real system this would query a database. Pre-fix it would happily return PII.
    return f"123 Main Street, nanny {nanny_id}'s home"

reveal_address = gated_tool("reveal_address", raw_reveal_address)

print("Pre-confirmation:", reveal_address(parent_id="p_01", nanny_id="n_01"))

booking_confirmed_for.add("p_01")
print("Post-confirmation:", reveal_address(parent_id="p_01", nanny_id="n_01"))

In [ ]:
# 5c. HUMAN-IN-THE-LOOP: before a booking-creating tool runs, surface a confirm prompt.
# In the notebook we simulate "YES" via a flag; in production this is a UI step.

def confirm_then_book(parent_id: str, nanny_id: str, day: str, hours: str, user_confirmed: bool):
    if not user_confirmed:
        return f"PENDING: Booking {nanny_id} for parent {parent_id} on {day} for {hours}h. Reply YES to confirm."
    # Now the action proceeds (e.g., calls draft_email, marks booking_confirmed_for)
    booking_confirmed_for.add(parent_id)
    return draft_email(parent_id=parent_id, nanny_id=nanny_id, day=day, hours=hours)

print("Without confirmation:")
print(confirm_then_book("p_02", "n_02", "thu", "6", user_confirmed=False))
print()
print("With confirmation:")
print(confirm_then_book("p_02", "n_02", "thu", "6", user_confirmed=True))

In [ ]:
# 🎯 TRY IT:
#   - Wrap draft_email itself with gated_tool requiring booking_confirmed_for. Then a
#     drift-confirmed agent can't accidentally send emails the parent didn't authorize.
#   - Add a "tool_audit_log" list that records every refusal — useful for monitoring.

# Demonstration: gated draft_email
gated_draft = gated_tool("draft_email", draft_email)
# Reset booking_confirmed_for for the demo
booking_confirmed_for.clear()
result = gated_draft(parent_id="p_03", nanny_id="n_01", day="thu", hours="6")
print("Gated (pre-confirm):", result[:120], "..." if len(result) > 120 else "")

## 6. Determinism + fallbacks

Two related but distinct properties:

- **Determinism** — same input → same output. Achieved with `temperature=0.0`, schema-locked outputs (BAML), and avoiding non-deterministic data (e.g., `random`, timestamps inside prompts).
- **Fallbacks** — when a tool fails, the agent shouldn't crash. It should retry, then degrade gracefully (e.g., "I can't access that right now; here's what I can do").

In [ ]:
# 6a. DETERMINISM: same input, same output. With temp=0 + BAML structured output,
# we can re-run the same decision and get the same answer.

decision_a = b.DecideOneTool(user_message="I need a Spanish-speaking nanny.", history="")
decision_b = b.DecideOneTool(user_message="I need a Spanish-speaking nanny.", history="")

print("Decision A thought:", decision_a.thought)
print("Decision B thought:", decision_b.thought)
print("Tool A:", decision_a.tool_call.name, dict(decision_a.tool_call.args))
print("Tool B:", decision_b.tool_call.name, dict(decision_b.tool_call.args))
print()
print("Same?", decision_a.thought == decision_b.thought)

In [ ]:
# 6b. FALLBACK: a tool that sometimes fails. We wrap it with retry + degradation.
import random

random.seed(0)
def flaky_search(query: str) -> list[dict]:
    if random.random() < 0.5:
        raise RuntimeError("upstream service unavailable")
    return search_nannies(query=query)

def with_retry(fn, attempts: int = 3, on_fail=None):
    def wrapped(**kwargs):
        last_exc = None
        for attempt in range(attempts):
            try:
                return fn(**kwargs)
            except Exception as e:  # noqa: BLE001
                last_exc = e
        # All attempts failed — return a graceful fallback message instead of raising.
        if on_fail is not None:
            return on_fail(kwargs, last_exc)
        return f"<<unavailable: {type(last_exc).__name__}>>"
    return wrapped

resilient_search = with_retry(
    flaky_search,
    attempts=3,
    on_fail=lambda args, exc: f"Search is temporarily unavailable. Please try again in a moment. ({type(exc).__name__})",
)

# Run several times to show the retry helps
for i in range(3):
    print(f"Attempt {i+1}: {resilient_search(query='CPR-certified nanny')}")

In [ ]:
# 6c. Plug the resilient tool into react_run. When the tool returns a string starting with
# "Search is temporarily unavailable", the agent sees that as the observation and can
# answer the user gracefully ("I can't find candidates right now — please retry later").

random.seed(7)  # try a seed where the first attempts often fail

tools_with_fallback = {
    "search_nannies": lambda query: resilient_search(query=query),
}

trace = react_run(
    user_message="I need a CPR-certified nanny.",
    decide_fn=decide_one_tool,
    tools=tools_with_fallback,
    max_steps=4,
)
for i, s in enumerate(trace.steps, 1):
    obs = str(s.observation)
    print(f"step {i}: {s.tool_name}({s.tool_args})")
    if obs and obs != "None":
        print(f"  obs: {obs[:140]}")
print(f"\nFinal: {trace.final_answer}")

In [ ]:
# 🎯 TRY IT:
#   - Change the flaky_search failure rate from 0.5 to 1.0 — the agent should fall back
#     to the "temporarily unavailable" message and finish with a graceful reply.
#   - Add backoff between retries: time.sleep(0.5 * (attempt+1)) inside with_retry.
#   - Try temperature=1.0 on DecideOneTool and re-run section 6a — outputs will diverge.

## 7. Multi-agent topology — Planner, Researcher, Executor

A single agent juggling planning, retrieval, and drafting can lose track of any one of them. Splitting roles makes each agent's job narrower and the trace easier to debug.

Our topology:

```
Planner ──▶ Researcher ──▶ Executor
   │           ▲              │
   │           │              ▼
   └────── (cited findings) — reply
```

- **Planner** decides who does what.
- **Researcher** cites sources (no fabrication).
- **Executor** drafts the user-facing reply from findings only.

Each agent is a separate BAML function in `baml_src/multi_agent.baml`.

In [ ]:
from nanny_workshop.agent import run_team

def planner(user_query: str):
    return b.PlanAgent(user_query=user_query)

def researcher(task: str, evidence_context: str):
    return b.ResearcherAgent(task=task, evidence_context=evidence_context)

def executor(task: str, findings: str):
    return b.ExecutorAgent(task=task, findings=findings)

def evidence_for(task: str) -> str:
    # Naive evidence selector: pass the full policy text for policy-related tasks,
    # otherwise a short summary of available nannies. In production you'd use
    # task-content classification to pick relevant evidence.
    policies_text = (ROOT / "data" / "policies" / "agency_policies.md").read_text()
    if any(w in task.lower() for w in ["policy", "cancel", "pii", "escalat", "safety"]):
        return policies_text
    # Otherwise return a compact nanny roster.
    import json as _json
    seed_data = _json.loads((ROOT / "data" / "seed_db.json").read_text())
    roster = [
        {"id": n["id"], "name": n["name"], "certifications": n["certifications"], "availability_days": n["availability_days"]}
        for n in seed_data["nannies"]
    ]
    return "Available nannies:\n" + _json.dumps(roster, indent=2)

team_result = run_team(
    user_query="What's your cancellation policy, and can you confirm my booking for Maria on Thursday for 6 hours?",
    plan_fn=planner,
    research_fn=researcher,
    executor_fn=executor,
    evidence_for_research=evidence_for,
)

print("PLAN:")
print(f"  goal: {team_result['plan'].goal}")
for step in team_result['plan'].steps:
    print(f"  - [{step.agent}] {step.task}")

print("\nFINDINGS:")
for f in team_result['findings']:
    print(f"  [{f.source}] {f.claim}")
    print(f"     excerpt: {f.excerpt[:200]}...")

print("\nREPLY:")
print(team_result["reply"])

In [ ]:
# 🎯 TRY IT:
#   - Open Phoenix UI: the trace tree now has 3 LLM spans per turn (planner, researcher, executor).
#   - Change the user query to "What's the cancellation fee for tomorrow's session?" —
#     the planner should produce a research-only step (no executor for booking-creation).
#   - Add a Critic step: call b.CriticAgent on the reply; if REVISE, re-run executor with the critique.